# Exercise 2 Code Generation with ReACT Prompting

**Tools used:** Google Colab, Python, Google AI Studio, the Gemini API, and the `google-genai` Python SDK.

**Goal:** Use a visible ReACT cycle: plan -> generate code -> run -> observe -> fix if necessary -> run again.

## Before running

Add `GEMINI_API_KEY` to Colab Secrets using the key icon on the left and enable notebook access. Do not paste the key into the notebook.

In [5]:
!pip -q install -U google-genai

from google import genai
from google.colab import userdata

try:
    api_key = userdata.get("GEMINI_API_KEY")
    if not api_key:
        raise ValueError
except Exception:
    raise ValueError(
        "GEMINI_API_KEY was not found. Add it under the key icon in Colab "
        "and turn on notebook access."
    )

client = genai.Client(api_key=api_key)
MODEL = "gemini-3.5-flash-lite"
print("Gemini client is ready. The API key was loaded from Colab Secrets.")

Gemini client is ready. The API key was loaded from Colab Secrets.


## Full ReACT prompt

```text
You are a Python coding assistant helping a beginner in Google Colab.
Follow a ReACT-style process and label your sections PLAN and CODE.

PLAN: Briefly explain how to calculate revenue per product from a list of
dictionaries containing product, units, and unit_price.

CODE: Return one complete Python program inside one ```python code block.
Use only the Python standard library. Print each product's revenue and total
revenue to two decimal places. Require units to be a nonnegative integer and
unit_price to be a nonnegative number. Raise clear ValueError messages for
missing fields or invalid values. Include normal data, a zero-unit row, and a
caught negative-unit test. Do not place code outside the code block.
```

In [6]:
REACT_PROMPT = "You are a Python coding assistant helping a beginner in Google Colab.\nFollow a ReACT-style process and label your sections PLAN and CODE.\n\nPLAN: Briefly explain how to calculate revenue per product from a list of\ndictionaries containing product, units, and unit_price.\n\nCODE: Return one complete Python program inside one ```python code block.\nUse only the Python standard library. Print each product's revenue and total\nrevenue to two decimal places. Require units to be a nonnegative integer and\nunit_price to be a nonnegative number. Raise clear ValueError messages for\nmissing fields or invalid values. Include normal data, a zero-unit row, and a\ncaught negative-unit test. Do not place code outside the code block."

first_response = client.models.generate_content(
    model=MODEL,
    contents=REACT_PROMPT,
).text

print("REASON AND GENERATE\n")
print(first_response)

REASON AND GENERATE

PLAN: We will iterate through a list of dictionaries representing products. For each dictionary, we validate that the required keys ("product", "units", "unit_price") are present and that their values are valid (units as a nonnegative integer, unit_price as a nonnegative number). We then calculate the revenue by multiplying units by unit_price. We will also include a try-except block to gracefully catch and display a `ValueError` when testing with a negative unit value.

CODE:
```python
def calculate_product_revenue(products):
    """Calculates revenue per product and total revenue with strict validation."""
    total_revenue = 0.0
    processed_products = []

    for item in products:
        # Check for missing fields
        if not all(k in item for k in ("product", "units", "unit_price")):
            raise ValueError(f"Missing required fields in item: {item}")

        product = item["product"]
        units = item["units"]
        unit_price = item["unit_pric

In [7]:
import re
import traceback

def extract_python(text):
    match = re.search(r"```python\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if not match:
        raise ValueError("Gemini did not return one Python code block.")
    return match.group(1).strip()

generated_code = extract_python(first_response)
print("RUN GENERATED CODE\n")

try:
    exec(generated_code, {})
    observation = "The generated program ran successfully without an uncaught error."
except Exception:
    observation = traceback.format_exc()

print("\nOBSERVE\n" + observation)

RUN GENERATED CODE

--- Running Normal & Zero-Unit Test ---
Product: Laptop | Revenue: $4999.95
Product: Mouse | Revenue: $637.50
Product: Keyboard | Revenue: $0.00
Total Revenue: $5637.45

--- Running Negative-Unit Test ---
Caught expected ValueError: Invalid units '-2' for product 'Monitor'. Must be a nonnegative integer.

OBSERVE
The generated program ran successfully without an uncaught error.


In [8]:
if observation.startswith("The generated program ran successfully"):
    final_code = generated_code
    print("FIX: No correction was necessary after the successful test.")
else:
    fix_prompt = f"""You are fixing Python code after observing an execution error.
Keep every requirement from the original request. Use the error to correct the
program. Return only the corrected program inside one ```python code block.

ORIGINAL REQUEST:
{REACT_PROMPT}

ORIGINAL CODE:
```python
{generated_code}
```

OBSERVED ERROR:
{observation}
"""
    fix_response = client.models.generate_content(
        model=MODEL,
        contents=fix_prompt,
    ).text
    print("FIX RESPONSE\n" + fix_response)
    final_code = extract_python(fix_response)

print("\nFINAL RUN\n")
exec(final_code, {})
print("\nReACT cycle completed successfully.")

FIX: No correction was necessary after the successful test.

FINAL RUN

--- Running Normal & Zero-Unit Test ---
Product: Laptop | Revenue: $4999.95
Product: Mouse | Revenue: $637.50
Product: Keyboard | Revenue: $0.00
Total Revenue: $5637.45

--- Running Negative-Unit Test ---
Caught expected ValueError: Invalid units '-2' for product 'Monitor'. Must be a nonnegative integer.

ReACT cycle completed successfully.


## Testing and iteration evidence

The notebook prints the generated plan and code, executes the program, and records the result under **OBSERVE**. If execution fails, the error and original code are sent back to Gemini for correction. The corrected program is then executed under **FINAL RUN**.